# <div align="center">🤗 Madar AI - Hugging Face LoRA Studio</div>
### <div align="center">Qwen2.5-Coder-1.5B Fine-Tuning for Ring Programming (Bilingual)</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azzeddine2017/Madar/blob/main/notebooks/Madar_LoRA_HuggingFace_Qwen_Coder.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar)
[![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Transformers%20%2B%20PEFT-yellow.svg)]()

---

This notebook trains **Qwen2.5-Coder-1.5B-Instruct** using the official **Hugging Face** ecosystem (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`) on the exact same **Madar Bilingual Ring Dataset** (`data/ring_instruct_bilingual.jsonl`).

### 🎯 Objectives:
1. **Reference Baseline**: Observe the official Hugging Face Loss curve on the bilingual Ring dataset.
2. **4-bit QLoRA**: Fast, memory-efficient training on a single Tesla T4 GPU (takes ~15-20 minutes).
3. **Response-Only Masking**: Only the assistant's Ring code and explanation are trained (using `DataCollatorForCompletionOnlyLM`).
4. **Loss Visualization**: Plot the loss across steps to analyze convergence.
5. **Export to GGUF**: Merge adapters and convert to `.gguf` for direct native execution in Madar.

## ⚙️ Step 1: Verify NVIDIA GPU
Verify that Google Colab is running with a GPU runtime (Tesla T4 / V100 / A100).

In [ ]:
!nvidia-smi

## 📦 Step 2: Clone Madar Repository & Install Hugging Face Tools
Fetches the bilingual dataset (`data/ring_instruct_bilingual.jsonl`) and installs the required libraries.

In [ ]:
import os

# 1. Clone repository to obtain the dataset
if not os.path.exists("/content/Madar/data"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git pull origin main

# 2. Uninstall conflicting torchaudio & install Hugging Face training stack
!pip uninstall -y -q torchaudio
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate matplotlib

## 🚀 Step 3: Train LoRA Model using Hugging Face SFTTrainer
Runs 4-bit QLoRA training on `Qwen2.5-Coder-1.5B-Instruct` across 3 epochs.

In [ ]:
import os, sys, json, torch

# Auto-recovery: Reload cached modules if Colab kernel had older packages in memory
try:
    from datasets import Dataset
except (ImportError, AttributeError):
    for mod in list(sys.modules.keys()):
        if mod.startswith("datasets") or mod.startswith("pyarrow"):
            del sys.modules[mod]
    try:
        from datasets import Dataset
    except (ImportError, AttributeError):
        print("\n[!] Restarting Colab session to apply updated packages. Please re-run this cell in 3 seconds!")
        os.kill(os.getpid(), 9)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_PATH = "data/ring_instruct_bilingual.jsonl"
OUTPUT_DIR = "adapters/qwen_hf_lora"

# 1. Load and format dataset (ChatML format with messages)
print(">>> Loading dataset...")
records = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

formatted_data = []
for r in records:
    instruction = r.get("instruction", "")
    inp = r.get("input", "")
    out = r.get("output", "")
    user_content = instruction + (f"\n{inp}" if inp.strip() else "")
    messages = [
        {"role": "system", "content": "You are an expert Ring programming assistant."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": out}
    ]
    formatted_data.append({"messages": messages})

hf_dataset = Dataset.from_list(formatted_data)
print(f"    Dataset ready with {len(hf_dataset)} instruction pairs.")

# 2. Load model in 4-bit NormalFloat (NF4)
print(">>> Loading 4-bit base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)
model.config.torch_dtype = torch.float16
model = prepare_model_for_kbit_training(model)

# 3. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# Ensure all LoRA layers and trainable parameters are float32 (fixes Tesla T4 GradScaler error)
for name, module in model.named_modules():
    if "lora" in name.lower():
        module.to(torch.float32)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

# 4. Custom Loss Tracker for Plotting
loss_history = []
class LossTrackerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            loss_history.append({"step": state.global_step, "loss": logs["loss"]})

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args,
    callbacks=[LossTrackerCallback()]
)

print(">>> Starting Training...")
trainer.train()

# 5. Save final LoRA adapters
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n[SUCCESS] LoRA Adapters saved to: {OUTPUT_DIR}")

## 📈 Step 4: Visualize the Hugging Face Loss Curve
Plots the training loss curve across steps to inspect convergence.

In [ ]:
import matplotlib.pyplot as plt

if loss_history:
    steps = [x["step"] for x in loss_history]
    losses = [x["loss"] for x in loss_history]
    
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, marker='o', color='#2563eb', label='Hugging Face Training Loss')
    plt.title("Qwen2.5-Coder-1.5B Bilingual LoRA Training Loss (Hugging Face)", fontsize=14)
    plt.xlabel("Global Step", fontsize=12)
    plt.ylabel("Cross-Entropy Loss", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig("hf_loss_curve.png", dpi=150)
    plt.show()
    print(f"Final Step: {steps[-1]} | Final Loss: {losses[-1]:.4f} | Min Loss: {min(losses):.4f}")
else:
    print("No loss history logged.")

## 🧪 Step 5: Test Model Inference (Bilingual Ring Generation)
Tests the fine-tuned model directly in Python on Ring coding prompts.

In [ ]:
import torch

# 1. Prepare model for high-performance inference
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

def test_model(prompt_text):
    messages = [
        {"role": "system", "content": "You are Madar AI, an expert code generator specialized in the Ring programming language."},
        {"role": "user", "content": prompt_text}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=384,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}\n[PROMPT]: {prompt_text}\n{'='*60}\n[RESPONSE]:\n{response}\n")

# Test 1: Arabic Prompt
test_model("اكتب دالة بلغة رينج لحساب مجموع عناصر مصفوفة مع مثال عملي")

# Test 2: English Prompt
test_model("Write a Ring code example demonstrating how to create a class with properties and methods.")

## 🔄 Step 6: Merge LoRA & Convert to GGUF for Native Madar
Merges the trained LoRA adapter back into full weights, then converts it to GGUF using `llama.cpp`.

In [ ]:
import os, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except:
    pass

OUTPUT_DIR = "adapters/qwen_hf_lora"
MERGED_DIR = "models/qwen2.5-coder-1.5b-ring-hf-merged"
os.makedirs("models", exist_ok=True)

# 1. Merge LoRA weights if not already merged
if not os.path.exists(os.path.join(MERGED_DIR, "model.safetensors")):
    print(">>> [1/3] Merging LoRA weights with base model in FP16...")
    try:
        del model
    except:
        pass
    gc.collect()
    torch.cuda.empty_cache()

    base_fp16 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    peft_merged = PeftModel.from_pretrained(base_fp16, OUTPUT_DIR)
    merged_model = peft_merged.merge_and_unload()

    merged_model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    print(f"    Merged model saved to: {MERGED_DIR}")
else:
    print(f">>> [1/3] Merged model already exists at: {MERGED_DIR} (Skipping re-merge)")

# 2. Patch tokenizer_config.json format for llama.cpp compatibility
tok_cfg_file = os.path.join(MERGED_DIR, "tokenizer_config.json")
if os.path.exists(tok_cfg_file):
    with open(tok_cfg_file, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    if "extra_special_tokens" in cfg and isinstance(cfg["extra_special_tokens"], list):
        cfg["extra_special_tokens"] = {t: t for t in cfg["extra_special_tokens"]}
        with open(tok_cfg_file, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)
        print("    Patched extra_special_tokens in tokenizer_config.json successfully.")

print("\n>>> [2/3] Setting up llama.cpp converter...")
if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
    !pip install -q -r llama.cpp/requirements.txt

print("\n>>> [3/3] Converting to GGUF (FP16)...")
GGUF_FP16 = "models/qwen2.5-coder-1.5b-ring-hf.gguf"
!python3 llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_FP16} --outtype f16

# Optional: Quantize to Q4_0
GGUF_Q4 = "models/qwen2.5-coder-1.5b-ring-hf-q4_0.gguf"
if not os.path.exists("llama.cpp/build/bin/llama-quantize"):
    !cd llama.cpp && cmake -B build && cmake --build build --config Release -t llama-quantize
!llama.cpp/build/bin/llama-quantize {GGUF_FP16} {GGUF_Q4} q4_0

print(f"\n[SUCCESS] Production GGUF model created at: {GGUF_Q4}")
print(">>> Model ready for download in Step 7!")


## 📥 Step 7: Download the GGUF Model to your Computer
Choose your preferred method to download the 900MB GGUF model:
- **Option A (Recommended):** Save directly to your Google Drive in 3 seconds.
- **Option B:** Generate a direct high-speed download link via `transfer.sh`.
- **Option C:** Browser direct download trigger.

In [ ]:
import os, shutil

GGUF_PATH = "models/qwen2.5-coder-1.5b-ring-hf-q4_0.gguf"
print(f"Target File: {GGUF_PATH} ({os.path.getsize(GGUF_PATH) / (1024*1024):.1f} MB)")

# -----------------------------------------------------
# OPTION A: Save directly to Google Drive (Best & Fast)
# -----------------------------------------------------
try:
    from google.colab import drive
    print("\n[Option A] Mounting Google Drive...")
    drive.mount('/content/drive')
    dest_drive = "/content/drive/MyDrive/qwen2.5-coder-1.5b-ring-hf-q4_0.gguf"
    shutil.copy(GGUF_PATH, dest_drive)
    print(f"[SUCCESS] Saved to Google Drive: {dest_drive}")
except Exception as e:
    print(f"Google Drive mount skipped/error: {e}")

# -----------------------------------------------------
# OPTION B: Generate Direct Download URL (transfer.sh)
# -----------------------------------------------------
print("\n[Option B] Generating High-Speed Direct Download Link...")
!curl -s --upload-file {GGUF_PATH} https://transfer.sh/qwen2.5-coder-1.5b-ring-hf-q4_0.gguf > /tmp/dl_link.txt
with open('/tmp/dl_link.txt', 'r') as f:
    dl_url = f.read().strip()
print(f"[DIRECT DOWNLOAD LINK]:\n{dl_url}")

# -----------------------------------------------------
# OPTION C: Browser File Trigger
# -----------------------------------------------------
# from google.colab import files
# files.download(GGUF_PATH)
